# Exact compressed posterior for finite acquisition designs

[Proof and assumptions](../11_exact_compressed_acquisition_posterior.md). This is a known-pole coefficient-measurement oracle, not a trained HSE or a hardware rate experiment. Full, diagonal and block summaries use different storage budgets. The real compressed conditional mixes compatible designs; it does not replace the unknown information matrix with its diagonal.

## 1. Realizable measurements and retained information
Four equally probable designs differ in within-mode and cross-mode information. All share a diagonal; two share each block summary.

In [ ]:
import numpy as np
from experiments.synthetic_known_pole.compression import (acquisition_family, prior_parameters,
    retained_information, matching_designs, conditional_posterior, evaluate_cell, _lognormal)
family=acquisition_family(.45)
print('Smallest information eigenvalue:', np.linalg.eigvalsh(family).min())
for method in ['diag','block','full']:
    group=matching_designs(family,retained_information(family[0],method),method)
    print(method, 'compatible designs:',group)
assert [len(matching_designs(family,retained_information(family[0],m),m))
        for m in ['diag','block','full']]==[4,2,1]

## 2. Verify weights through observation space
The decoder sees b, so the design weight must use p(b|d) including the change-of-variables determinant.

In [ ]:
b=np.array([[2.,.3,-1.,.4]])
prior=prior_parameters('gaussian')
p=conditional_posterior(b,family,np.arange(4),prior)
direct=[]
for J in family:
    A=np.linalg.cholesky(J).T
    x=np.linalg.solve(A.T,b.T).T
    direct.append(_lognormal(x,np.zeros(4),A@A.T+np.eye(4))[0]-np.linalg.slogdet(A)[1])
direct=np.exp(np.array(direct)-np.max(direct)); direct/=direct.sum()
np.testing.assert_allclose(np.exp(p.log_weights[0]),direct,atol=1e-12)
assert np.ptp(direct)>.05
print('Conditional design probabilities:',direct)
plugin=conditional_posterior(b,np.diag(np.diag(family[0]))[None],np.array([0]),prior)
assert abs(p.log_prob(b)[0]-plugin.log_prob(b)[0])>.01
print('Exact-mixture / diagonal-plug-in log density:',p.log_prob(b),plugin.log_prob(b))

## 3. Sufficiency controls
Blocks retain all information when cross-mode coupling is absent. Full operator side input closes every arm's Bayes gap. Hidden cross-mode coupling leaves a residual.

In [ ]:
for prior_name in ['gaussian','mixture']:
    no_cross=evaluate_cell(prior_name,0.,17,128)
    np.testing.assert_allclose(no_cross['coarse','block_exact']['total_gap_nats'],0,atol=1e-12)
    with_cross=evaluate_cell(prior_name,.45,17,128)
    for (regime,name),r in with_cross.items():
        np.testing.assert_allclose(r['total_gap_nats'],r['compression_nats']+r['fitting_nats'],atol=1e-12)
        if regime=='full_operator':
            np.testing.assert_allclose(r['total_gap_nats'],0,atol=1e-12)
    assert with_cross['coarse','block_exact']['epsilon_predictor_gap'].mean()>1e-3
    print(prior_name, 'coarse diagonal / block gaps:',
          with_cross['coarse','diag_exact']['total_gap_nats'].mean(),
          with_cross['coarse','block_exact']['total_gap_nats'].mean())

## 4. Admission boundary
The formula makes Task B measurable but does not prove a same-budget learned HSE gain. A finite mixture already represents this oracle posterior. Run the full experiment with `python -m experiments.synthetic_known_pole.run_compression --events-per-seed 2048 --seeds 0 1 2 --bootstrap 1000 --output-dir outputs/task_b`. See paper/results.md for paired intervals and the non-monotonic coupling curve. Negative log-ratio estimates are never clipped.

In [ ]:
print("THEORY_DEMO_PASS::11_exact_compressed_acquisition_posterior")